# Lecture 5 — Loss Functions and Optimization
## Lab Notebook · Deep Learning · UCU

---

### Overview
In Notebook 3, we explored **deep neural networks** and saw that depth dramatically increases expressiveness per parameter. But we used default settings for everything else — Adam optimizer, default initialization, cross-entropy loss. Where do these choices come from? Do they matter?

In this notebook, we peel back those defaults. From Lecture 5, we know that **loss functions** arise from the maximum likelihood recipe: choose a probability distribution over outputs, and minimize the negative log-likelihood. We also learned that **optimizers** vary widely in how they navigate the loss landscape, and that **initialization** can make or break training in deep networks.

By the end of this notebook, you will:
1. **Derive** loss functions from the maximum likelihood recipe and implement them from scratch
2. **Inspect** gradients computed by backpropagation and verify them manually
3. **Compare** SGD, SGD with momentum, and Adam on the same architecture
4. **Diagnose** vanishing/exploding gradients and fix them with He initialization
5. **Experiment** with learning rates, batch sizes, and learning rate schedules

### Useful References

| Resource | Link |
|----------|------|
| PyTorch `torch.optim` | [pytorch.org/docs/stable/optim.html](https://pytorch.org/docs/stable/optim.html) |
| PyTorch `torch.nn.init` | [pytorch.org/docs/stable/nn.init.html](https://pytorch.org/docs/stable/nn.init.html) |
| PyTorch Loss Functions | [pytorch.org/docs/stable/nn.html#loss-functions](https://pytorch.org/docs/stable/nn.html#loss-functions) |
| FashionMNIST dataset | [github.com/zalandoresearch/fashion-mnist](https://github.com/zalandoresearch/fashion-mnist) |
| UCU Deep Learning — Lecture 5 notes | [../../lectures/lecture 5/notes.md](../../lectures/lecture%205/notes.md) |
| *Understanding Deep Learning* — Prince (2023) | Chapters 5, 6, 7 |

In [ ]:
# Setup — run this cell first
try:
    import google.colab
    IN_COLAB = True
    !pip install -q torchinfo
except ImportError:
    IN_COLAB = False

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR, CosineAnnealingLR
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms
from torchinfo import summary
import numpy as np
import matplotlib.pyplot as plt
import copy
import time
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# UCU color palette
C1, C2, C3, C4, C5 = '#19326E', '#50ACB0', '#CD742A', '#A3477F', '#907FAB'
C6, C7 = '#4294CC', '#89A943'

In [ ]:
# Load FashionMNIST — shared across all sections
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training samples : {len(train_dataset)}")
print(f"Test samples     : {len(test_dataset)}")
print(f"Input shape      : {train_dataset[0][0].shape}")
print(f"Number of classes : {len(class_names)}")

---

## 1. Loss Functions from Maximum Likelihood

From Lecture 5, we know that loss functions aren't arbitrary choices — they follow from a **recipe**:

1. Choose a probability distribution $Pr(\mathbf{y}|\boldsymbol{\theta})$ over the output domain
2. Let the network predict the distribution parameters: $\boldsymbol{\theta} = \mathbf{f}[\mathbf{x}, \boldsymbol{\phi}]$
3. Minimize the **negative log-likelihood**: $L[\boldsymbol{\phi}] = -\sum_i \log Pr(\mathbf{y}_i | \mathbf{f}[\mathbf{x}_i, \boldsymbol{\phi}])$

This recipe gives us **least squares** (from a Gaussian), **binary cross-entropy** (from a Bernoulli), and **multiclass cross-entropy** (from a categorical distribution). In this section, you'll implement each one from scratch and verify they match PyTorch's built-in versions.

### Exercise 1.1 — Mean Squared Error from scratch

For regression with Gaussian-distributed outputs, the negative log-likelihood reduces to the **least squares loss**:

$$L[\boldsymbol{\phi}] = \frac{1}{I}\sum_{i=1}^{I}(y_i - f[\mathbf{x}_i, \boldsymbol{\phi}])^2$$

Implement this loss function using basic PyTorch operations (no loops). Then verify it matches `nn.MSELoss()`.

In [ ]:
def mse_loss_manual(y_pred, y_true):
    """Compute mean squared error loss.
    
    Args:
        y_pred: predictions, shape (N,) or (N, 1)
        y_true: targets, shape (N,) or (N, 1)
    Returns:
        scalar MSE loss
    """
    # SOLUTION
    return ((y_pred - y_true) ** 2).mean()


# Verify against PyTorch
torch.manual_seed(42)
y_pred = torch.randn(32)
y_true = torch.randn(32)

our_loss = mse_loss_manual(y_pred, y_true)
pytorch_loss = nn.MSELoss()(y_pred, y_true)

print(f"Our MSE loss    : {our_loss.item():.6f}")
print(f"PyTorch MSE loss: {pytorch_loss.item():.6f}")
assert torch.allclose(our_loss, pytorch_loss, atol=1e-6), "Losses don't match!"
print("✓ Losses match!")

### Exercise 1.2 — Multiclass Cross-Entropy from scratch

For classification with $K$ classes, the network outputs $K$ raw **logits** $\mathbf{z} \in \mathbb{R}^K$. The **softmax** function converts these to probabilities:

$$\text{softmax}_k(\mathbf{z}) = \frac{\exp(z_k)}{\sum_{k'=1}^K \exp(z_{k'})}$$

The **cross-entropy loss** for a single sample with true class $c$ is:

$$\ell = -\log(\text{softmax}_c(\mathbf{z})) = -z_c + \log\sum_{k'=1}^K \exp(z_{k'})$$

Implement this in three steps:
1. Apply softmax to the logits
2. Select the probability of the correct class for each sample
3. Take negative log, average over the batch

Then verify it matches `nn.CrossEntropyLoss()`.

In [ ]:
def cross_entropy_manual(logits, targets):
    """Compute multiclass cross-entropy loss from raw logits.
    
    Args:
        logits: raw network outputs, shape (N, K)
        targets: integer class labels, shape (N,)
    Returns:
        scalar cross-entropy loss (mean over batch)
    """
    # SOLUTION
    # Step 1: apply softmax
    probs = torch.softmax(logits, dim=1)
    # Step 2: select probability of correct class
    correct_probs = probs[torch.arange(len(targets)), targets]
    # Step 3: negative log, average
    return -torch.log(correct_probs).mean()


# Verify against PyTorch
torch.manual_seed(42)
logits = torch.randn(32, 10)  # batch of 32, 10 classes
targets = torch.randint(0, 10, (32,))

our_loss = cross_entropy_manual(logits, targets)
pytorch_loss = nn.CrossEntropyLoss()(logits, targets)

print(f"Our CE loss    : {our_loss.item():.6f}")
print(f"PyTorch CE loss: {pytorch_loss.item():.6f}")
assert torch.allclose(our_loss, pytorch_loss, atol=1e-5), "Losses don't match!"
print("✓ Losses match!")

### Exercise 1.3 — Binary Cross-Entropy from scratch

For binary classification ($y \in \{0, 1\}$), the network outputs a single logit $z \in \mathbb{R}$. The **sigmoid** maps it to a probability:

$$\sigma(z) = \frac{1}{1 + \exp(-z)}$$

The **binary cross-entropy** loss is:

$$\ell = -(1 - y)\log(1 - \sigma(z)) - y\log(\sigma(z))$$

Implement this and verify against `nn.BCEWithLogitsLoss()`.

In [ ]:
def bce_loss_manual(logits, targets):
    """Compute binary cross-entropy loss from raw logits.
    
    Args:
        logits: raw network outputs, shape (N,)
        targets: binary labels (0 or 1), shape (N,)
    Returns:
        scalar BCE loss (mean over batch)
    """
    # SOLUTION
    probs = torch.sigmoid(logits)
    eps = 1e-7
    loss = -(targets * torch.log(probs + eps) + (1 - targets) * torch.log(1 - probs + eps))
    return loss.mean()


# Verify against PyTorch
torch.manual_seed(42)
logits = torch.randn(32)
targets = torch.randint(0, 2, (32,)).float()

our_loss = bce_loss_manual(logits, targets)
pytorch_loss = nn.BCEWithLogitsLoss()(logits, targets)

print(f"Our BCE loss    : {our_loss.item():.6f}")
print(f"PyTorch BCE loss: {pytorch_loss.item():.6f}")
assert torch.allclose(our_loss, pytorch_loss, atol=1e-4), "Losses don't match!"
print("✓ Losses match!")

In [ ]:
# Visualization: softmax in action on real FashionMNIST samples
# Pass a few test images through a random (untrained) network and show predicted probabilities

demo_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))

# Get 4 random test images
torch.manual_seed(0)
indices = torch.randint(0, len(test_dataset), (4,))
images = torch.stack([test_dataset[i][0] for i in indices])
labels = [test_dataset[i][1] for i in indices]

with torch.no_grad():
    logits = demo_model(images)
    probs = torch.softmax(logits, dim=1)

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i in range(4):
    # Show image
    axes[0, i].imshow(images[i].squeeze(), cmap='gray')
    axes[0, i].set_title(f'True: {class_names[labels[i]]}', fontsize=10)
    axes[0, i].axis('off')
    
    # Show softmax probabilities
    bars = axes[1, i].bar(range(10), probs[i].numpy(), color=C1, alpha=0.7)
    bars[labels[i]].set_color(C2)  # highlight correct class
    axes[1, i].set_xticks(range(10))
    axes[1, i].set_xticklabels([n[:6] for n in class_names], rotation=45, ha='right', fontsize=7)
    axes[1, i].set_ylabel('Probability')
    axes[1, i].set_ylim(0, 1)
    axes[1, i].grid(True, alpha=0.3, axis='y')

plt.suptitle('Softmax Output: Raw Logits → Class Probabilities (untrained network)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

> **Question 1.1** — PyTorch's `CrossEntropyLoss` takes **raw logits**, not probabilities. Why? What numerical problem would arise if you applied softmax first and then took the log separately? 
>
> *Hint:* think about what happens when one logit is much larger than the others.
>
> *Answer:* When one logit is very large, softmax produces a value very close to 1.0 (or 0.0 for other classes). Taking log(1.0) is fine, but log(0.0) = -infinity. By combining softmax + log into a single `log_softmax` operation, PyTorch uses the **log-sum-exp trick** to compute this stably: log(softmax_k(z)) = z_k - log(sum(exp(z))). This avoids ever computing a probability that could be exactly 0.

> **Question 1.2** — From the lecture: least squares loss assumes Gaussian-distributed outputs, and cross-entropy assumes categorical outputs. What would happen if you used **MSE loss for classification** (with one-hot encoded targets)? We'll test this empirically in Section 3 — write down your prediction now.
>
> *Answer:* MSE loss should still work for classification (it provides gradients that push outputs toward the one-hot target), but it will likely converge slower and reach lower accuracy. MSE treats all output dimensions independently and doesn't account for the constraint that class probabilities should sum to 1. The gradients from MSE are also less informative — for a confident wrong prediction, cross-entropy produces strong gradients while MSE's gradients can be small due to saturation.

---

## 2. Gradient Inspection and Backpropagation

From Lecture 5, we know that **backpropagation** efficiently computes gradients of the loss with respect to every parameter via the chain rule. PyTorch's `autograd` handles this automatically — but it's important to understand what it's actually computing.

In this section, you'll build a tiny network with known weights, run a forward and backward pass, and verify the gradients by hand.

### Exercise 2.1 — Inspect gradients on a tiny network

Build a small network with **fixed, known weights**: `nn.Linear(2, 3)` → `nn.ReLU()` → `nn.Linear(3, 1)`. Set the weights manually (code provided below). Then:
1. Run a forward pass with input `x = [1.0, 2.0]` and target `y = 1.0`
2. Compute MSE loss
3. Call `.backward()`
4. Print the gradient (`.grad`) for every parameter using `named_parameters()`

In [ ]:
# Build tiny network with known weights
tiny_net = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
)

# Set weights manually for reproducibility
with torch.no_grad():
    # Layer 0 (Linear 2→3): W shape (3,2), b shape (3,)
    tiny_net[0].weight.copy_(torch.tensor([[ 0.5, -0.3],
                                            [ 0.2,  0.8],
                                            [-0.4,  0.1]]))
    tiny_net[0].bias.copy_(torch.tensor([0.1, -0.2, 0.3]))
    
    # Layer 2 (Linear 3→1): W shape (1,3), b shape (1,)
    tiny_net[2].weight.copy_(torch.tensor([[0.6, -0.5, 0.4]]))
    tiny_net[2].bias.copy_(torch.tensor([0.1]))

x = torch.tensor([[1.0, 2.0]])  # single input
y = torch.tensor([[1.0]])        # target

# SOLUTION
pred = tiny_net(x)
loss = nn.MSELoss()(pred, y)
loss.backward()

print(f"Input:      {x}")
print(f"Prediction: {pred.item():.4f}")
print(f"Target:     {y.item():.4f}")
print(f"MSE Loss:   {loss.item():.4f}")
print()
for name, param in tiny_net.named_parameters():
    print(f"{name:>10s}  |  shape {str(list(param.shape)):>10s}  |  grad: {param.grad}")

### Exercise 2.2 — Verify a gradient by hand

Let's trace the computation manually for the **output layer bias** $b_2$. The forward pass computes:

$$\mathbf{f}_0 = W_1 \mathbf{x} + \mathbf{b}_1 \quad \rightarrow \quad \mathbf{h}_1 = \text{ReLU}(\mathbf{f}_0) \quad \rightarrow \quad \hat{y} = W_2 \mathbf{h}_1 + b_2$$

The MSE loss is $L = (\hat{y} - y)^2$. By the chain rule:

$$\frac{\partial L}{\partial b_2} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial b_2} = 2(\hat{y} - y) \cdot 1$$

Compute this value numerically using the intermediate values from your forward pass. Does it match the autograd gradient?

In [ ]:
# SOLUTION
# Step 1: compute forward pass manually
W1 = tiny_net[0].weight.data  # (3, 2)
b1 = tiny_net[0].bias.data    # (3,)
W2 = tiny_net[2].weight.data  # (1, 3)
b2 = tiny_net[2].bias.data    # (1,)

x_flat = x.squeeze()  # (2,)
f0 = W1 @ x_flat + b1
print(f"Pre-activation f0: {f0}")

h1 = torch.relu(f0)
print(f"After ReLU h1:     {h1}")

y_hat = W2 @ h1 + b2
print(f"Prediction y_hat:  {y_hat.item():.4f}")

# Step 2: compute dL/db2
dL_db2 = 2 * (y_hat.item() - y.item())
print(f"\nHand-computed dL/db2: {dL_db2:.4f}")
print(f"Autograd dL/db2:      {tiny_net[2].bias.grad.item():.4f}")
print(f"Match: {abs(dL_db2 - tiny_net[2].bias.grad.item()) < 1e-5}")

> **Question 2.1** — Look at the gradient magnitudes for each parameter in the tiny network. Which layer has the largest gradients? Which has the smallest? What does this suggest about how information flows backward through the network?
>
> *Answer:* The output layer (layer 2) typically has the largest gradients because it's closest to the loss. The input layer (layer 0) has smaller gradients because the error signal must pass through ReLU and the second linear layer, attenuating at each step. This illustrates why deep networks can suffer from vanishing gradients — the error signal weakens as it propagates backward through many layers.

### Exercise 2.3 — Gradient norms across layers in a deeper network

Now let's scale up. Build a 5-layer network (width 64, ReLU activations), forward-pass a batch of 256 random inputs, compute a cross-entropy loss against random labels, and record the **mean gradient norm** of each weight matrix after `.backward()`.

This will give us our first look at how gradients behave in deeper networks — a preview of Section 4.

In [ ]:
# SOLUTION
torch.manual_seed(42)

# 1. Build network
deep_net = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 64), nn.ReLU(),
    nn.Linear(64, 64),  nn.ReLU(),
    nn.Linear(64, 64),  nn.ReLU(),
    nn.Linear(64, 64),  nn.ReLU(),
    nn.Linear(64, 64),  nn.ReLU(),
    nn.Linear(64, 10)
)

# 2. Forward pass
x_rand = torch.randn(256, 1, 28, 28)
y_rand = torch.randint(0, 10, (256,))

# 3. Loss + backward
logits = deep_net(x_rand)
loss = nn.CrossEntropyLoss()(logits, y_rand)
loss.backward()

# 4. Collect gradient norms
grad_norms = []
for m in deep_net:
    if isinstance(m, nn.Linear):
        grad_norms.append(m.weight.grad.norm().item())

for i, gn in enumerate(grad_norms):
    print(f"Layer {i}: gradient norm = {gn:.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(grad_norms)), grad_norms, color=C1)
ax.set_xlabel('Layer index')
ax.set_ylabel('Weight gradient norm')
ax.set_title('Gradient Norms Across Layers (5-layer network, default init)')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

> **Question 2.2** — Do you see a trend in gradient magnitudes across layers? Do gradients grow or shrink as you go from the output layer toward the input? What does this imply for training the early layers of a deep network?
>
> *Answer:* With default initialization, gradients tend to shrink (or grow unpredictably) as we go from the output toward the input. The early layers receive the weakest gradients, which means they learn the slowest. In extreme cases (vanishing gradients), the early layers barely update at all, effectively making the deep network behave like a shallow one. This is exactly the problem that He initialization addresses, as we'll see in Section 4.

---

## 3. The Optimizer Showdown

From Lecture 5, we learned three optimizers of increasing sophistication:

| Optimizer | Update rule | Key idea |
|-----------|------------|----------|
| **SGD** | $\boldsymbol{\phi} \leftarrow \boldsymbol{\phi} - \alpha \nabla L$ | Follow the (noisy) gradient |
| **SGD + Momentum** | Accumulate exponential moving average of gradients | Smooth out oscillations |
| **Adam** | Adaptive per-parameter learning rates + momentum | Robust to learning rate choice |

But how different are they in practice? In this section, you'll train the **same architecture** with each optimizer and systematically vary the learning rate. This is the kind of experiment you'll do routinely when developing real models.

We use a 4-hidden-layer network with ~25K parameters (similar to Notebook 3), with **He initialization** applied from the start (we'll understand *why* in Section 4).

In [ ]:
# Shared architecture and utilities for all experiments in this section

def make_model():
    """Create a fresh 4-hidden-layer network with He initialization (~25K params).
    Architecture: 784 → 48 → 48 → 48 → 48 → 10
    """
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 48), nn.ReLU(),
        nn.Linear(48, 48),  nn.ReLU(),
        nn.Linear(48, 48),  nn.ReLU(),
        nn.Linear(48, 48),  nn.ReLU(),
        nn.Linear(48, 10)
    )
    # Apply He initialization
    for m in model:
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight)
            m.bias.data.fill_(0.0)
    return model

# Verify parameter count
info = summary(make_model(), input_size=(1, 1, 28, 28), verbose=0)
print(f"Model parameters: {info.total_params:,}")


# Training utilities (same pattern as Notebooks 2-3)
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch. Returns (avg_loss, accuracy %)."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / len(loader), 100. * correct / total


def evaluate(model, loader, criterion, device):
    """Evaluate model. Returns (avg_loss, accuracy %)."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = criterion(logits, y)
            total_loss += loss.item()
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / len(loader), 100. * correct / total


def run_experiment(model_fn, optimizer_fn, num_epochs, train_loader, test_loader,
                   criterion=None, device=device):
    """Train a model and return history dict.
    
    Args:
        model_fn: callable that returns a fresh model
        optimizer_fn: callable that takes model.parameters() and returns an optimizer
        num_epochs: number of epochs to train
        train_loader, test_loader: data loaders
        criterion: loss function (default: CrossEntropyLoss)
        device: torch device
    Returns:
        dict with keys: train_loss, test_loss, train_acc, test_acc (lists)
    """
    if criterion is None:
        criterion = nn.CrossEntropyLoss()
    
    model = model_fn().to(device)
    optimizer = optimizer_fn(model.parameters())
    
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}
    for epoch in range(num_epochs):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        te_loss, te_acc = evaluate(model, test_loader, criterion, device)
        history['train_loss'].append(tr_loss)
        history['test_loss'].append(te_loss)
        history['train_acc'].append(tr_acc)
        history['test_acc'].append(te_acc)
    
    return history

### Experiment 3.1 — Optimizer type comparison

Train the same architecture with three optimizers, all using `lr=0.01`, for **15 epochs**. Use the `run_experiment` helper — you just need to define the optimizer functions.

In [ ]:
NUM_EPOCHS_3 = 15

# SOLUTION
optimizers = {
    'SGD':          lambda params: optim.SGD(params, lr=0.01),
    'SGD+Momentum': lambda params: optim.SGD(params, lr=0.01, momentum=0.9),
    'Adam':         lambda params: optim.Adam(params, lr=0.01),
}

results_3_1 = {}
for name, opt_fn in optimizers.items():
    results_3_1[name] = run_experiment(make_model, opt_fn, NUM_EPOCHS_3,
                                        train_loader, test_loader)
    print(f"{name:>15s}  |  Test acc: {results_3_1[name]['test_acc'][-1]:.1f}%")

In [ ]:
# Visualization: 2x2 grid of training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_opt = [C1, C2, C3]
epochs_range = range(1, NUM_EPOCHS_3 + 1)

for (name, hist), color in zip(results_3_1.items(), colors_opt):
    axes[0, 0].plot(epochs_range, hist['train_loss'], color=color, linewidth=2, label=name)
    axes[0, 1].plot(epochs_range, hist['test_loss'],  color=color, linewidth=2, label=name)
    axes[1, 0].plot(epochs_range, hist['train_acc'],  color=color, linewidth=2, label=name)
    axes[1, 1].plot(epochs_range, hist['test_acc'],   color=color, linewidth=2, label=name)

titles = ['Train Loss', 'Test Loss', 'Train Accuracy (%)', 'Test Accuracy (%)']
for ax, title in zip(axes.flat, titles):
    ax.set_xlabel('Epoch'); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('Experiment 3.1: Optimizer Comparison (lr=0.01)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

> **Question 3.1** — Comparing the three optimizers at lr=0.01:
> 1. Which optimizer converges fastest (reaches high accuracy in the fewest epochs)?
> 2. Which reaches the best final test accuracy?
> 3. Does momentum help SGD? By how much?
>
> *Answer:* Adam typically converges fastest and reaches the best accuracy within 15 epochs. SGD+Momentum is a clear improvement over vanilla SGD — momentum smooths out the noisy gradient updates and helps traverse flat or elongated regions of the loss landscape. Vanilla SGD at lr=0.01 is the slowest, often needing many more epochs. The improvement from momentum is typically 5-10% in final accuracy at this epoch count.

### Experiment 3.2 — Learning rate sensitivity

Now we test how sensitive each optimizer is to the learning rate. For each of the three optimizers, train with learning rates `[0.0001, 0.001, 0.01, 0.1]`. That's **12 training runs** — this may take a few minutes.

Use 10 epochs per run to save time.

In [ ]:
NUM_EPOCHS_LR = 10
learning_rates = [0.0001, 0.001, 0.01, 0.1]

# SOLUTION
optimizer_configs = {
    'SGD':          lambda params, lr: optim.SGD(params, lr=lr),
    'SGD+Momentum': lambda params, lr: optim.SGD(params, lr=lr, momentum=0.9),
    'Adam':         lambda params, lr: optim.Adam(params, lr=lr),
}

results_3_2 = {}
for opt_name, opt_factory in optimizer_configs.items():
    results_3_2[opt_name] = {}
    for lr in learning_rates:
        opt_fn = lambda params, _lr=lr, _f=opt_factory: _f(params, _lr)
        results_3_2[opt_name][lr] = run_experiment(
            make_model, opt_fn, NUM_EPOCHS_LR, train_loader, test_loader)
        print(f"{opt_name:>15s}  lr={lr:.4f}  |  Test: {results_3_2[opt_name][lr]['test_acc'][-1]:.1f}%")

In [ ]:
# Visualization: 3x1 grid — one subplot per optimizer, curves for each learning rate
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
lr_colors = [C5, C6, C2, C3]
epochs_range = range(1, NUM_EPOCHS_LR + 1)

for ax, (opt_name, lr_results) in zip(axes, results_3_2.items()):
    for (lr, hist), color in zip(lr_results.items(), lr_colors):
        ax.plot(epochs_range, hist['test_acc'], color=color, linewidth=2, label=f'lr={lr}')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(opt_name); ax.legend(); ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 95)

plt.suptitle('Experiment 3.2: Learning Rate Sensitivity by Optimizer', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

> **Question 3.2** — Analyzing learning rate sensitivity:
> 1. Which optimizer is **most sensitive** to the learning rate choice? Which is **most robust**?
> 2. Why does Adam tolerate a wider range of learning rates? Connect your answer to the lecture's explanation of per-parameter adaptive rates (the division by $\sqrt{\tilde{\mathbf{v}}}$).
>
> *Answer:* Vanilla SGD is the most sensitive — too small and it barely moves, too large and it diverges. Adam is the most robust because it normalizes each parameter's update by $\sqrt{\tilde{\mathbf{v}}}$ (the running average of squared gradients). This effectively makes each parameter's step size approximately constant regardless of gradient magnitude, so the actual learning rate matters less. SGD+Momentum is in between.

> **Question 3.3** — What happens with `lr=0.1` for SGD without momentum? For Adam? Explain the difference in behavior.
>
> *Answer:* SGD with lr=0.1 often diverges or oscillates wildly because the raw gradients are multiplied by a large step size, causing overshooting. Adam with lr=0.1 may also struggle, but the adaptive denominator dampens the effective step size — each parameter moves by approximately ±lr regardless of gradient magnitude. So Adam at lr=0.1 is more stable than SGD at the same rate, though it may still be suboptimal.

### Experiment 3.3 — Cross-Entropy vs MSE for classification

In Question 1.2, you predicted what would happen if we used MSE loss for classification. Let's test it.

Using the best optimizer and learning rate from the experiments above, train two models for 15 epochs:
- One with `nn.CrossEntropyLoss()` (the correct choice from maximum likelihood)
- One with `nn.MSELoss()` using **one-hot encoded targets**

> **Note:** To use MSE for classification, you need to convert integer labels to one-hot vectors. We provide a helper and a wrapper loss below.

In [ ]:
# Helper: MSE loss that accepts integer labels (converts to one-hot internally)
class MSEClassificationLoss(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.num_classes = num_classes
        self.mse = nn.MSELoss()
    
    def forward(self, logits, targets):
        one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
        return self.mse(logits, one_hot)


# SOLUTION
results_3_3 = {}
results_3_3['CrossEntropy'] = run_experiment(
    make_model, lambda p: optim.Adam(p, lr=0.001), 15,
    train_loader, test_loader, criterion=nn.CrossEntropyLoss())
print(f"CrossEntropy  |  Test acc: {results_3_3['CrossEntropy']['test_acc'][-1]:.1f}%")

results_3_3['MSE'] = run_experiment(
    make_model, lambda p: optim.Adam(p, lr=0.001), 15,
    train_loader, test_loader, criterion=MSEClassificationLoss())
print(f"MSE           |  Test acc: {results_3_3['MSE']['test_acc'][-1]:.1f}%")

In [ ]:
# Visualization: CE vs MSE comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
loss_colors = [C1, C3]
epochs_range = range(1, 16)

for (name, hist), color in zip(results_3_3.items(), loss_colors):
    axes[0].plot(epochs_range, hist['test_acc'], color=color, linewidth=2, label=name)
    axes[1].plot(epochs_range, hist['train_acc'], '--', color=color, linewidth=1.5, alpha=0.5, label=f'{name} (train)')
    axes[1].plot(epochs_range, hist['test_acc'], color=color, linewidth=2, label=f'{name} (test)')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Test Accuracy: CE vs MSE'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Train vs Test Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Experiment 3.3: Cross-Entropy vs MSE for Classification', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

for name, hist in results_3_3.items():
    print(f"{name:>14s}  |  Final test acc: {hist['test_acc'][-1]:.1f}%")

> **Question 3.4** — Comparing cross-entropy vs MSE for classification:
> 1. How does MSE compare to cross-entropy in convergence speed and final accuracy?
> 2. Did this match your prediction from Question 1.2?
> 3. Why might cross-entropy be preferred even if MSE eventually works? Think about the gradient behavior: for cross-entropy, the gradient is proportional to $(p_k - y_k)$; for MSE applied to logits, what happens to the gradient when the logits are far from the one-hot target?
>
> *Answer:* Cross-entropy typically converges faster and reaches higher accuracy. MSE works but is slower because: (1) MSE gradients for classification are less efficient — they don't account for the softmax structure, treating each output independently; (2) Cross-entropy's gradient is simply (predicted probability - target), which is large when the model is wrong and small when it's right. MSE on raw logits can have small gradients when outputs are far from target but saturated, leading to slow learning. This confirms the maximum likelihood recipe — using the correct distributional assumption gives the best loss function.

---

## 4. Initialization Matters

In Section 3, we quietly used **He initialization** (`kaiming_normal_`) without explaining why. Now let's understand what happens without it.

From Lecture 5, each layer transforms the variance of activations by a factor of $\frac{1}{2}D_h\sigma_\Omega^2$ (the $\frac{1}{2}$ comes from ReLU clipping half the values). If this factor is:
- **Less than 1** ($\sigma_\Omega^2 < 2/D_h$): activations shrink layer by layer → **vanishing gradients**
- **Greater than 1** ($\sigma_\Omega^2 > 2/D_h$): activations grow layer by layer → **exploding gradients**
- **Equal to 1** ($\sigma_\Omega^2 = 2/D_h$): activations stay stable → **He initialization**

Let's make this concrete with a 20-layer network.

### Exercise 4.1 — Activation variance propagation

Build a 20-layer ReLU network (width 100, no training). Initialize the weights with three different scales:
1. `nn.init.normal_(w, std=0.01)` — too small
2. `nn.init.normal_(w, std=1.0)` — too large
3. `nn.init.kaiming_normal_(w)` — He initialization

For each initialization, forward-pass a batch of 256 random inputs and record the **variance of activations** at each layer. Plot all three on one figure (use log scale for the y-axis).

In [ ]:
def make_deep_net(num_layers=20, width=100):
    """Create a deep ReLU network (no Flatten — expects flat input)."""
    layers = [nn.Linear(width, width), nn.ReLU()]
    for _ in range(num_layers - 1):
        layers += [nn.Linear(width, width), nn.ReLU()]
    return nn.Sequential(*layers)


def init_weights(model, mode='he'):
    """Initialize all Linear layers with a given mode."""
    for m in model:
        if isinstance(m, nn.Linear):
            if mode == 'small':
                nn.init.normal_(m.weight, std=0.01)
            elif mode == 'large':
                nn.init.normal_(m.weight, std=1.0)
            elif mode == 'he':
                nn.init.kaiming_normal_(m.weight)
            m.bias.data.fill_(0.0)


def measure_activation_variance(model, x):
    """Forward pass, recording activation variance after each ReLU.
    
    Returns:
        list of variance values, one per layer
    """
    variances = []
    h = x
    for layer in model:
        h = layer(h)
        if isinstance(layer, nn.ReLU):
            variances.append(h.var().item())
    return variances


# SOLUTION
torch.manual_seed(42)
x_init = torch.randn(256, 100)

act_variances = {}
for mode in ['small', 'large', 'he']:
    net = make_deep_net(num_layers=20, width=100)
    init_weights(net, mode=mode)
    with torch.no_grad():
        act_variances[mode] = measure_activation_variance(net, x_init)

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
init_colors = {'small': C4, 'large': C3, 'he': C2}
init_labels = {'small': 'std=0.01 (too small)', 'large': 'std=1.0 (too large)', 'he': 'He init (just right)'}

for mode, variances in act_variances.items():
    ax.plot(range(1, len(variances) + 1), variances, 'o-',
            color=init_colors[mode], linewidth=2, markersize=4, label=init_labels[mode])

ax.set_xlabel('Layer'); ax.set_ylabel('Activation Variance (log scale)')
ax.set_title('Activation Variance Across 20 Layers')
ax.set_yscale('log'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

> **Question 4.1** — Analyzing the activation variance plot:
> 1. What happens to the activation variance with `std=0.01`? With `std=1.0`? Describe the trend.
> 2. Why does He initialization keep the variance approximately stable?
> 3. Use the formula from the lecture: $\text{Var}(f') = \frac{1}{2}D_h \sigma_\Omega^2 \cdot \text{Var}(f)$. For `width=100`, what value of $\sigma_\Omega^2$ makes this ratio equal to 1? Verify this matches what `kaiming_normal_` uses.
>
> *Answer:* 
> 1. With std=0.01: variance shrinks exponentially toward zero — activations vanish. With std=1.0: variance explodes exponentially — activations blow up to NaN.
> 2. He initialization sets $\sigma_\Omega^2 = 2/D_h$, which makes $\frac{1}{2}D_h\sigma_\Omega^2 = \frac{1}{2} \cdot 100 \cdot \frac{2}{100} = 1$. Each layer preserves the variance.
> 3. $\sigma_\Omega^2 = 2/D_h = 2/100 = 0.02$, so $\sigma_\Omega = \sqrt{0.02} \approx 0.1414$. This is exactly what `kaiming_normal_` uses for ReLU (fan_in mode).

### Exercise 4.2 — Gradient variance propagation

Now let's look at the backward pass. Using the same three initializations, compute a loss on random data, call `.backward()`, and record the **gradient norm of each weight matrix**. This mirrors the activation analysis but for gradients.

In [ ]:
def measure_gradient_norms(model, x, y):
    """Forward + backward pass, recording weight gradient norms per layer.
    
    Returns:
        list of gradient norms, one per Linear layer (from input to output)
    """
    criterion = nn.MSELoss()
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()
    
    grad_norms = []
    for m in model:
        if isinstance(m, nn.Linear):
            grad_norms.append(m.weight.grad.norm().item())
    return grad_norms


# SOLUTION
torch.manual_seed(42)
x_init = torch.randn(256, 100)
y_init = torch.randn(256, 100)

grad_norms_by_init = {}
for mode in ['small', 'large', 'he']:
    net = make_deep_net(num_layers=20, width=100)
    init_weights(net, mode=mode)
    grad_norms_by_init[mode] = measure_gradient_norms(net, x_init, y_init)

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))

for mode, norms in grad_norms_by_init.items():
    ax.plot(range(1, len(norms) + 1), norms, 'o-',
            color=init_colors[mode], linewidth=2, markersize=4, label=init_labels[mode])

ax.set_xlabel('Layer (from input to output)')
ax.set_ylabel('Weight Gradient Norm (log scale)')
ax.set_title('Gradient Norms Across 20 Layers')
ax.set_yscale('log'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

> **Question 4.2** — Comparing activation and gradient propagation:
> 1. Do the gradient norms mirror the activation variance pattern? For which initialization do gradients vanish? Explode?
> 2. What would happen if you tried to train the `std=0.01` network? Would the early layers learn anything? Why or why not?
>
> *Answer:* Yes, the gradient pattern mirrors activations: with std=0.01, gradients vanish (especially in early layers), and with std=1.0, they explode. He initialization keeps gradients stable across layers. Training the std=0.01 network would effectively freeze the early layers — their gradients are so small that parameter updates are negligible. The network would behave as if only the last few layers are trainable, wasting the capacity of the earlier layers.

### Exercise 4.3 — Training impact of initialization

Now let's see how initialization affects actual training. Using the best optimizer and learning rate from Section 3, train the 4-hidden-layer FashionMNIST model with three initialization schemes:
1. `std=0.01` (too small)
2. `std=1.0` (too large)
3. He initialization

15 epochs each. Also try this with a shallower 1-layer network to see if initialization matters equally for shallow vs deep architectures.

In [ ]:
def make_model_with_init(mode='he'):
    """Create the 4-layer model with a specific initialization."""
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 48), nn.ReLU(),
        nn.Linear(48, 48),  nn.ReLU(),
        nn.Linear(48, 48),  nn.ReLU(),
        nn.Linear(48, 48),  nn.ReLU(),
        nn.Linear(48, 10)
    )
    for m in model:
        if isinstance(m, nn.Linear):
            if mode == 'small':
                nn.init.normal_(m.weight, std=0.01)
            elif mode == 'large':
                nn.init.normal_(m.weight, std=1.0)
            elif mode == 'he':
                nn.init.kaiming_normal_(m.weight)
            m.bias.data.fill_(0.0)
    return model


def make_shallow_with_init(mode='he'):
    """Create a 1-layer model with a specific initialization."""
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 256), nn.ReLU(),
        nn.Linear(256, 10)
    )
    for m in model:
        if isinstance(m, nn.Linear):
            if mode == 'small':
                nn.init.normal_(m.weight, std=0.01)
            elif mode == 'large':
                nn.init.normal_(m.weight, std=1.0)
            elif mode == 'he':
                nn.init.kaiming_normal_(m.weight)
            m.bias.data.fill_(0.0)
    return model


# SOLUTION
init_modes = ['small', 'large', 'he']
results_4_3_deep = {}
results_4_3_shallow = {}

for mode in init_modes:
    results_4_3_deep[mode] = run_experiment(
        lambda m=mode: make_model_with_init(m),
        lambda p: optim.Adam(p, lr=0.001), 15, train_loader, test_loader)
    results_4_3_shallow[mode] = run_experiment(
        lambda m=mode: make_shallow_with_init(m),
        lambda p: optim.Adam(p, lr=0.001), 15, train_loader, test_loader)
    print(f"Init={mode:>5s}  |  Deep: {results_4_3_deep[mode]['test_acc'][-1]:.1f}%  "
          f"|  Shallow: {results_4_3_shallow[mode]['test_acc'][-1]:.1f}%")

In [ ]:
# Visualization: side-by-side comparison — deep vs shallow
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, 16)

for (mode, hist), color in zip(results_4_3_deep.items(), [C4, C3, C2]):
    axes[0].plot(epochs_range, hist['test_acc'], color=color, linewidth=2, label=init_labels[mode])
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Deep Network (4 hidden layers)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

for (mode, hist), color in zip(results_4_3_shallow.items(), [C4, C3, C2]):
    axes[1].plot(epochs_range, hist['test_acc'], color=color, linewidth=2, label=init_labels[mode])
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Shallow Network (1 hidden layer)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Exercise 4.3: Effect of Initialization — Deep vs Shallow', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

> **Question 4.3** — Initialization and network depth:
> 1. Does initialization matter more for the **deep** or **shallow** network? Compare the gap between the best and worst initialization for each.
> 2. Why does depth amplify the effect of bad initialization? Think about how the multiplication factor $\frac{1}{2}D_h\sigma_\Omega^2$ compounds across layers.
> 3. Based on everything you've seen in this section, would you ever train a deep network without He initialization? Why or why not?
>
> *Answer:*
> 1. Initialization matters much more for the deep network. The shallow network is relatively robust — even with bad initialization, one hidden layer doesn't compound the variance problem much. The deep network shows a large gap between initializations.
> 2. The factor $\frac{1}{2}D_h\sigma_\Omega^2$ is applied at **each** layer. After $K$ layers, the variance is scaled by $(\frac{1}{2}D_h\sigma_\Omega^2)^K$. If this factor is even slightly different from 1, raising it to a high power makes it either exponentially small or large. For example, with std=0.01 and width 48: $\frac{1}{2}(48)(0.0001) = 0.0024$, so after 4 layers the variance is scaled by $0.0024^4 \approx 3 \times 10^{-11}$.
> 3. No — He initialization is simple, costs nothing, and prevents vanishing/exploding gradients. There is no reason not to use it for ReLU networks.

---

## 5. Batch Size Effects

From Lecture 5, we know that **SGD** estimates the gradient using a random mini-batch. The batch size controls the noise level:
- **Small batches** → noisier gradients → more exploration, potential regularization effect
- **Large batches** → smoother gradients → faster per epoch (better GPU utilization), but may converge to sharper minima

Let's test this empirically.

### Exercise 5.1 — Batch size sweep

Train the best configuration from Section 3 (best optimizer, best lr, He init) with batch sizes `[32, 128, 256, 1024]`. Use 15 epochs.

You'll need to create **new DataLoaders** with each batch size.

In [ ]:
batch_sizes = [32, 128, 256, 1024]

# SOLUTION
results_5 = {}
for bs in batch_sizes:
    bs_train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True)
    bs_test_loader  = DataLoader(test_dataset,  batch_size=bs, shuffle=False)
    results_5[bs] = run_experiment(
        make_model, lambda p: optim.Adam(p, lr=0.001), 15,
        bs_train_loader, bs_test_loader)
    print(f"Batch size {bs:>5d}  |  Test acc: {results_5[bs]['test_acc'][-1]:.1f}%")

In [ ]:
# Visualization: accuracy curves + final accuracy bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bs_colors = [C1, C2, C3, C4]
epochs_range = range(1, 16)

for (bs, hist), color in zip(results_5.items(), bs_colors):
    axes[0].plot(epochs_range, hist['test_acc'], color=color, linewidth=2, label=f'bs={bs}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Test Accuracy by Batch Size'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

final_accs = [results_5[bs]['test_acc'][-1] for bs in batch_sizes]
bars = axes[1].bar([str(bs) for bs in batch_sizes], final_accs, color=bs_colors)
axes[1].set_xlabel('Batch Size'); axes[1].set_ylabel('Final Test Accuracy (%)')
axes[1].set_title('Final Test Accuracy')
axes[1].grid(True, alpha=0.3, axis='y')
for bar, acc in zip(bars, final_accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{acc:.1f}%', ha='center', fontsize=10)

plt.suptitle('Experiment 5.1: Batch Size Effects', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

> **Question 5.1** — Analyzing batch size effects:
> 1. How does batch size affect **convergence speed** (how fast accuracy rises in early epochs)?
> 2. How does batch size affect **final accuracy**? Do smaller batches generalize better?
> 3. The lecture says smaller batches add noise that helps escape local minima. Do you see evidence of this in your results?
>
> *Answer:* Smaller batch sizes (32, 128) typically see faster convergence in terms of epochs because each epoch involves more parameter updates (more mini-batches per epoch). However, each epoch takes longer in wall-clock time. Smaller batches often generalize slightly better due to the regularization effect of noisy gradients — the noise helps the optimizer avoid sharp minima and find flatter ones that generalize better. The effect is usually modest (1-2%) on FashionMNIST.

> **Question 5.2** — Larger batches are faster per epoch (better GPU utilization) but may need more epochs. Smaller batches see more parameter updates per epoch. If you had a fixed **wall-clock time budget**, what batch size would you choose and why?
>
> *Answer:* In practice, batch size 128-256 is a good default for FashionMNIST-sized datasets. It balances GPU utilization with enough noise for good generalization. Very small batches (32) waste GPU compute, while very large batches (1024+) may need learning rate scaling tricks and more epochs. The "linear scaling rule" suggests increasing lr proportionally to batch size, but that's beyond this notebook.

---

## Optional: Learning Rate Schedules

In all experiments so far, we've used a **constant learning rate**. But the lecture mentions that it's common practice to **decrease the learning rate** over time: start with a high rate for fast exploration, then reduce it for fine-tuning near the optimum.

PyTorch provides several scheduling strategies in `torch.optim.lr_scheduler`. Two popular ones:
- **StepLR**: multiply the learning rate by `gamma` every `step_size` epochs
- **CosineAnnealingLR**: smoothly decay the learning rate following a cosine curve from the initial value to near zero

### Exercise O.1 — Comparing learning rate schedules

Train the best configuration from Section 3 with three strategies:
1. **Constant** learning rate (baseline)
2. **StepLR**: halve the learning rate every 5 epochs (`step_size=5, gamma=0.5`)
3. **CosineAnnealingLR**: smooth cosine decay over all epochs (`T_max=NUM_EPOCHS`)

Use **20 epochs** to give the schedules time to show their effect.

> **Note:** Schedulers need to be called once per epoch with `scheduler.step()`. We provide a modified training loop below.

In [ ]:
def run_experiment_with_scheduler(model_fn, optimizer_fn, scheduler_fn, num_epochs,
                                  train_loader, test_loader, device=device):
    """Train a model with a learning rate scheduler.
    
    Args:
        model_fn: callable returning a fresh model
        optimizer_fn: callable taking model.parameters(), returning optimizer
        scheduler_fn: callable taking optimizer, returning scheduler (or None for constant lr)
        num_epochs: number of epochs
    Returns:
        dict with train_loss, test_loss, train_acc, test_acc, lr_history
    """
    criterion = nn.CrossEntropyLoss()
    model = model_fn().to(device)
    optimizer = optimizer_fn(model.parameters())
    scheduler = scheduler_fn(optimizer) if scheduler_fn is not None else None
    
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': [], 'lr': []}
    for epoch in range(num_epochs):
        current_lr = optimizer.param_groups[0]['lr']
        history['lr'].append(current_lr)
        
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        te_loss, te_acc = evaluate(model, test_loader, criterion, device)
        
        if scheduler is not None:
            scheduler.step()
        
        history['train_loss'].append(tr_loss)
        history['test_loss'].append(te_loss)
        history['train_acc'].append(tr_acc)
        history['test_acc'].append(te_acc)
    
    return history


NUM_EPOCHS_OPT = 20

# SOLUTION
schedule_configs = {
    'Constant':  None,
    'StepLR':    lambda opt: StepLR(opt, step_size=5, gamma=0.5),
    'Cosine':    lambda opt: CosineAnnealingLR(opt, T_max=NUM_EPOCHS_OPT),
}

results_opt = {}
for name, sched_fn in schedule_configs.items():
    results_opt[name] = run_experiment_with_scheduler(
        make_model,
        lambda p: optim.Adam(p, lr=0.001),
        sched_fn,
        NUM_EPOCHS_OPT, train_loader, test_loader)
    print(f"{name:>10s}  |  Test acc: {results_opt[name]['test_acc'][-1]:.1f}%")

In [ ]:
# Visualization: LR schedule + test accuracy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sched_colors = [C1, C2, C3]
epochs_range = range(1, NUM_EPOCHS_OPT + 1)

# Learning rate over time
for (name, hist), color in zip(results_opt.items(), sched_colors):
    axes[0].plot(epochs_range, hist['lr'], color=color, linewidth=2, label=name)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Learning Rate')
axes[0].set_title('Learning Rate Schedule'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Test accuracy
for (name, hist), color in zip(results_opt.items(), sched_colors):
    axes[1].plot(epochs_range, hist['test_acc'], color=color, linewidth=2, label=name)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Optional: Learning Rate Schedules', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

for name, hist in results_opt.items():
    print(f"{name:>10s}  |  Final test acc: {hist['test_acc'][-1]:.1f}%")

> **Question O.1** — Learning rate schedules:
> 1. Does scheduling improve final accuracy compared to a constant learning rate?
> 2. Which schedule works better — StepLR or CosineAnnealing? At what point during training does the benefit become visible?
> 3. Why might reducing the learning rate late in training help? Think about the loss landscape near a minimum.
>
> *Answer:* Learning rate scheduling can modestly improve final accuracy (typically 0.5-1%). The benefit becomes visible in the later epochs — early on, all strategies perform similarly since the learning rate hasn't decayed much yet. Reducing the learning rate late in training helps because near a minimum, large updates cause the optimizer to oscillate around the optimum rather than converging to it. A smaller step size allows finer-grained adjustments. CosineAnnealing provides a smooth decay which can work well, while StepLR creates discrete drops that may cause brief instability after each step.

### Exercise O.2 — Your best FashionMNIST configuration

Combine the best choices from all sections — optimizer, learning rate, initialization, batch size, and learning rate schedule — into one final training run. Use 20 epochs and report your best test accuracy.

This is the best you've achieved on FashionMNIST so far in the course!

In [ ]:
# SOLUTION: combining best choices
# Based on experiments: Adam, lr=0.001, He init, batch_size=256, CosineAnnealing

best_train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
best_test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

best_result = run_experiment_with_scheduler(
    make_model,
    lambda p: optim.Adam(p, lr=0.001),
    lambda opt: CosineAnnealingLR(opt, T_max=20),
    20, best_train_loader, best_test_loader)

print(f"\nBest configuration final test accuracy: {best_result['test_acc'][-1]:.1f}%")
print(f"Best train accuracy: {best_result['train_acc'][-1]:.1f}%")